# HWP 5.x 바이너리 파싱 과정 (Raw)

`udfp` 패키지 import 없이, `olefile` + `struct` + `zlib`만으로 HWP 파일을 처음부터 파싱합니다.

**사용 라이브러리:** `olefile` (OLE2 컨테이너), `struct` (바이너리), `zlib` (압축 해제) — 전부 외부 or 표준 라이브러리

In [1]:
import struct
import zlib
import os
from collections import Counter

import olefile

# ── 분석 대상 파일 (여기만 바꾸면 됨) ──
HWP_PATH = os.path.abspath("../workspace/10_인라인혼합서식.hwp")

print(f"파일: {os.path.basename(HWP_PATH)}")
print(f"크기: {os.path.getsize(HWP_PATH):,} bytes")

파일: 10_인라인혼합서식.hwp
크기: 13,312 bytes


---
## 1. OLE2 컨테이너 열기

HWP 5.x = Microsoft Compound File (OLE2).  
안에 `FileHeader`, `DocInfo`, `BodyText/Section0` 등의 스트림이 있음.

In [ ]:
ole = olefile.OleFileIO(HWP_PATH)

print("=== 스트림 목록 ===")
for path in ole.listdir(streams=True, storages=False):
    print(f"  {path}")
    raw = ole.openstream(path).read()
    print(f"  {'/'.join(path):30s}  {len(raw):>6,} bytes")

=== 스트림 목록 ===
  HwpSummaryInformation             457 bytes
  BodyText/Section0                  458 bytes
  DocInfo                          1,362 bytes
  DocOptions/_LinkDoc                524 bytes
  FileHeader                         256 bytes
  PrvImage                         5,126 bytes
  PrvText                              4 bytes
  Scripts/DefaultJScript              16 bytes
  Scripts/JScriptVersion              13 bytes


---
## 2. FileHeader 분석 — 압축 플래그 확인

FileHeader offset 36에 4바이트 flags:  
- bit0: 스트림 압축 여부  
- bit1: 암호화  
- bit2: 배포 제한

In [2]:
fh = ole.openstream("FileHeader").read()

print(f"FileHeader 크기: {len(fh)} bytes")
print(f"시그니처 (0-31): {fh[:32]}")
print(f"  → 텍스트: {fh[:32].decode('ascii', errors='replace')}")

flags = struct.unpack_from('<I', fh, 36)[0]
compressed = bool(flags & 0x01)

print(f"\nflags = 0x{flags:08X}")
print(f"  bit0 압축:     {compressed}")
print(f"  bit1 암호화:   {bool(flags & 0x02)}")
print(f"  bit2 배포제한: {bool(flags & 0x04)}")

NameError: name 'ole' is not defined

In [ ]:
def read_stream(ole, path, compressed):
    """스트림을 읽고, compressed=True이면 raw deflate(wbits=-15)로 해제."""
    raw = ole.openstream(path).read()
    if compressed and path != "FileHeader":
        return zlib.decompress(raw, wbits=-15)
    return raw

# DocInfo, BodyText/Section0 읽기
docinfo_bytes = read_stream(ole, "DocInfo", compressed)
body_bytes = read_stream(ole, "BodyText/Section0", compressed)

print(f"DocInfo:           {len(docinfo_bytes):>6,} bytes (압축 해제 후)")
print(f"BodyText/Section0: {len(body_bytes):>6,} bytes (압축 해제 후)")

---
## 3. HWPTAG 레코드 체인 디코딩

DocInfo와 BodyText 모두 같은 형식의 레코드 체인:

```
┌─ 4B 헤더 ──────────────────────┐
│ tag_id (10bit) │ level (10bit) │ size (12bit) │
└────────────────────────────────┘
│ payload (size bytes)           │
└────────────────────────────────┘
```

size == 0xFFF이면 다음 4바이트가 실제 size (확장 헤더).

In [ ]:
def parse_records(stream):
    """바이트 스트림에서 (tag_id, level, payload, offset) 튜플 리스트를 반환."""
    records = []
    off = 0
    n = len(stream)
    while off + 4 <= n:
        rec_start = off
        h = struct.unpack_from('<I', stream, off)[0]
        tag_id = h & 0x3FF          # 하위 10비트
        level  = (h >> 10) & 0x3FF  # 중간 10비트
        size   = (h >> 20) & 0xFFF  # 상위 12비트
        off += 4
        if size == 0xFFF:  # 확장 크기
            if off + 4 > n:
                break
            size = struct.unpack_from('<I', stream, off)[0]
            off += 4
        if off + size > n:
            break
        records.append((tag_id, level, stream[off:off+size], rec_start))
        off += size
    return records

# 태그 이름 사전
TAG = {
    16: "DOCUMENT_PROPERTIES", 17: "ID_MAPPINGS", 18: "BIN_DATA",
    19: "FACE_NAME", 20: "BORDER_FILL", 21: "CHAR_SHAPE",
    22: "TAB_DEF", 23: "NUMBERING", 24: "BULLET",
    25: "PARA_SHAPE", 26: "STYLE", 27: "DOC_DATA",
    66: "PARA_HEADER", 67: "PARA_TEXT", 68: "PARA_CHAR_SHAPE",
    69: "PARA_LINE_SEG", 70: "PARA_RANGE_TAG",
    71: "CTRL_HEADER", 72: "LIST_HEADER", 73: "PAGE_DEF",
    77: "TABLE",
}

def tname(tid):
    return TAG.get(tid, f"TAG_{tid}")

print("함수 정의 완료.")

In [ ]:
docinfo_recs = parse_records(docinfo_bytes)
body_recs = parse_records(body_bytes)

print(f"DocInfo 레코드: {len(docinfo_recs)}개")
print(f"Body    레코드: {len(body_recs)}개")

print("\n=== DocInfo 태그별 카운트 ===")
for name, cnt in Counter(tname(r[0]) for r in docinfo_recs).most_common():
    print(f"  {name:25s} × {cnt}")

print("\n=== Body 태그별 카운트 ===")
for name, cnt in Counter(tname(r[0]) for r in body_recs).most_common():
    print(f"  {name:25s} × {cnt}")

In [ ]:
# BodyText 레코드 트리 시각화 (level로 들여쓰기)
print("=== BodyText 레코드 트리 ===")
for tag_id, level, payload, offset in body_recs:
    indent = "  " * level
    extra = ""
    if tag_id == 71:  # CTRL_HEADER — ctrlId는 첫 4바이트를 역순으로 읽으면 ASCII
        ctrl_id = payload[:4][::-1].decode('ascii', errors='replace') if len(payload) >= 4 else '?'
        extra = f"  ctrl_id='{ctrl_id}'"
    print(f"{indent}[L{level}] {tname(tag_id):20s} {len(payload):>4}B  @{offset}{extra}")

---
## 4. DocInfo — FaceName 파싱

FACE_NAME(tag=19) 레코드 구조:
```
offset 0: flags (1B)  — bit0=HAS_FONT_NAME
offset 1: name_len (uint16 LE) — UTF-16LE 문자 수
offset 3: name (name_len × 2 bytes) — UTF-16LE
```

In [ ]:
face_names = []

for tag_id, level, payload, _ in docinfo_recs:
    if tag_id != 19:  # FACE_NAME
        continue
    
    flags = payload[0]
    name = None
    if flags & 0x01 and len(payload) >= 3:
        name_len = struct.unpack_from('<H', payload, 1)[0]
        name_bytes = payload[3 : 3 + name_len * 2]
        name = name_bytes.decode('utf-16-le', errors='replace')
    face_names.append(name)

print(f"=== FaceName ({len(face_names)}개) ===")
for i, fn in enumerate(face_names):
    print(f"  [{i:2d}] {fn}")

---
## 5. DocInfo — CharShape 파싱

CHAR_SHAPE(tag=21) — 74바이트 구조체:

```
 0-13   face_id[7]: uint16[7]     ← 한글/영문/한자/일본어/기타/기호/사용자
14-20   ratio[7]: uint8[7]        ← 장평 (기본 100)
21-27   spacing[7]: int8[7]       ← 자간
28-34   rel_size[7]: uint8[7]
35-41   offset[7]: int8[7]
42-45   base_size: int32          ← 1pt = 100 HWPUNIT
46-49   attr: uint32
           bit0 = italic
           bit1 = bold
           bits2-3 = underline_type
           bits18-20 = strikethrough
52-55   text_color: uint32        ← 0x00BBGGRR
56-59   underline_color: uint32
```

In [ ]:
char_shapes = []

for tag_id, level, pay, _ in docinfo_recs:
    if tag_id != 21:  # CHAR_SHAPE
        continue
    
    cs = {}
    if len(pay) >= 14:
        face_ids = struct.unpack_from('<7H', pay, 0)
        cs['hangul_face_id'] = face_ids[0]
        cs['latin_face_id'] = face_ids[1]
    
    if len(pay) >= 46:
        cs['base_size'] = struct.unpack_from('<i', pay, 42)[0]
        cs['font_size_pt'] = cs['base_size'] / 100
    
    if len(pay) >= 50:
        attr = struct.unpack_from('<I', pay, 46)[0]
        cs['italic']        = bool(attr & 0x01)
        cs['bold']          = bool(attr & 0x02)
        cs['underline_type']= (attr >> 2) & 0x03
        cs['underline']     = cs['underline_type'] > 0
        cs['strikethrough'] = ((attr >> 18) & 0x07) >= 2
        cs['attr_raw']      = f"0x{attr:08X}"
    
    if len(pay) >= 56:
        c = struct.unpack_from('<I', pay, 52)[0]
        cs['color'] = f"#{(c>>16)&0xFF:02x}{(c>>8)&0xFF:02x}{c&0xFF:02x}" if c else None
    
    char_shapes.append(cs)

print(f"=== CharShape ({len(char_shapes)}개) ===")
for i, cs in enumerate(char_shapes):
    fid = cs.get('hangul_face_id', '?')
    font = face_names[fid] if isinstance(fid, int) and fid < len(face_names) else '?'
    size = cs.get('font_size_pt', '?')
    flags = []
    if cs.get('bold'): flags.append('B')
    if cs.get('italic'): flags.append('I')
    if cs.get('underline'): flags.append('U')
    if cs.get('strikethrough'): flags.append('S')
    if cs.get('color'): flags.append(cs['color'])
    f = ', '.join(flags) if flags else 'plain'
    print(f"  [{i:2d}] {font:16s} {size:>5}pt  [{f}]  attr={cs.get('attr_raw', '?')}")

---
## 6. DocInfo — ParaShape 파싱

PARA_SHAPE(tag=25):
```
 0-3   attr: uint32
         bits 0-1: line_spacing_type
         bits 2-4: alignment (0=left, 1=right, 2=center, 3=justify)
 4-27  uint32[6]: left_m, right_m, top_sp, bot_sp, indent, line_spacing
```

In [ ]:
ALIGN_MAP = {0: 'left', 1: 'right', 2: 'center', 3: 'justify', 4: 'justify', 5: 'justify'}
LS_TYPE_MAP = {0: 'ratio', 1: 'fixed', 2: 'leading_only', 3: 'minimum'}

para_shapes = []

for tag_id, level, pay, _ in docinfo_recs:
    if tag_id != 25:  # PARA_SHAPE
        continue
    
    ps = {}
    if len(pay) >= 4:
        attr = struct.unpack_from('<I', pay, 0)[0]
        ps['alignment'] = ALIGN_MAP.get((attr >> 2) & 0x7, 'left')
        ps['ls_type']   = LS_TYPE_MAP.get(attr & 0x3, 'ratio')
    if len(pay) >= 28:
        left, right, top, bot, indent, ls = struct.unpack_from('<6I', pay, 4)
        ps['left_margin']  = left
        ps['right_margin'] = right
        ps['space_before'] = top
        ps['space_after']  = bot
        ps['first_indent'] = indent
        ps['line_spacing'] = ls
    para_shapes.append(ps)

print(f"=== ParaShape ({len(para_shapes)}개) ===")
for i, ps in enumerate(para_shapes):
    print(f"  [{i:2d}] align={ps.get('alignment','?'):8s}  "
          f"ls={ps.get('line_spacing', 0)} ({ps.get('ls_type','?')})  "
          f"margins=L{ps.get('left_margin',0)}/R{ps.get('right_margin',0)}")

---
## 7. DocInfo — Style 파싱

STYLE(tag=26):
```
BSTR(이름) → BSTR(영문이름) → type(1B) → next_style(1B)
→ lang_id(2B) → lock_mask(2B) → para_shape_id(2B) → char_shape_id(2B)
```
BSTR = uint16(문자수) + UTF-16LE 데이터

In [ ]:
styles = []

def read_bstr(pay, off):
    """BSTR: uint16 char_count + UTF-16LE → (문자열, 다음 오프셋)"""
    if off + 2 > len(pay):
        return None, off
    n = struct.unpack_from('<H', pay, off)[0]
    off += 2
    text = pay[off:off + n*2].decode('utf-16-le', errors='replace') if n > 0 else ''
    return text, off + n * 2

for tag_id, level, pay, _ in docinfo_recs:
    if tag_id != 26:  # STYLE
        continue
    
    name, off = read_bstr(pay, 0)
    ename, off = read_bstr(pay, off)
    
    s = {'name': name, 'ename': ename}
    if off + 6 <= len(pay):
        s['type'] = 'paragraph' if pay[off] == 0 else 'character'
        off += 6  # type(1) + next_style(1) + lang_id(2) + lock_mask(2)
    if off + 4 <= len(pay):
        ps_id, cs_id = struct.unpack_from('<HH', pay, off)
        s['para_shape_id'] = ps_id
        s['char_shape_id'] = cs_id
    styles.append(s)

print(f"=== Style ({len(styles)}개) ===")
for i, s in enumerate(styles):
    ps_id = s.get('para_shape_id', '?')
    cs_id = s.get('char_shape_id', '?')
    print(f"  [{i:2d}] {s.get('name','?'):20s} type={s.get('type','?'):12s} "
          f"ps={ps_id}, cs={cs_id}")

---
## 8. BodyText — PARA_TEXT 텍스트 추출

PARA_TEXT(tag=67) 페이로드 = UTF-16LE 문자 + 제어 코드 혼재:

| 코드 범위 | 바이트 소비 | 의미 |
|-----------|------------|------|
| `> 0x001F` | 2B | 일반 문자 |
| `0x0000, 0x0009, 0x000A, 0x000D` | 2B | 단순 제어 (탭, 줄바꿈 등) |
| 나머지 `≤ 0x001F` | 16B | 인라인 오브젝트 (2B 코드 + 14B 파라미터) |

In [ ]:
CTRL_2BYTE = {0x0000, 0x0009, 0x000A, 0x000D}

def extract_text(pt_payload):
    """PARA_TEXT 페이로드에서 일반 문자만 추출. 제어 코드 건너뜀."""
    chars = []
    off = 0
    n = len(pt_payload)
    while off + 2 <= n:
        code = struct.unpack_from('<H', pt_payload, off)[0]
        if code > 0x001F:
            if code != 0xFFFF:
                chars.append(chr(code))
            off += 2
        elif code in CTRL_2BYTE:
            off += 2
        else:
            off += 16  # 인라인 오브젝트
    return ''.join(chars)

# 모든 PARA_TEXT 추출
para_texts = [(pay, offset) for tid, lvl, pay, offset in body_recs if tid == 67]

print(f"PARA_TEXT 레코드: {len(para_texts)}개\n")
for i, (pay, _) in enumerate(para_texts):
    text = extract_text(pay)
    print(f"  [{i}] ({len(pay):>3}B) \"{text}\"")

In [ ]:
# 첫 번째 비어있지 않은 PARA_TEXT의 바이트 단위 디코딩 추적
for pay, _ in para_texts:
    if len(pay) < 4:
        continue
    text = extract_text(pay)
    if not text.strip():
        continue
    
    print(f"=== 바이트 단위 디코딩 (payload {len(pay)}B) ===")
    print(f"추출 결과: \"{text}\"\n")
    
    off = 0
    n = len(pay)
    step = 0
    while off + 2 <= n:
        code = struct.unpack_from('<H', pay, off)[0]
        if code > 0x001F:
            print(f"  @{off:3d}: 0x{code:04X} → '{chr(code)}'")
            off += 2
        elif code in CTRL_2BYTE:
            label = {0: 'NULL', 9: 'TAB', 0xA: 'LF', 0xD: 'CR'}.get(code, '?')
            print(f"  @{off:3d}: 0x{code:04X} → [{label}] (2B skip)")
            off += 2
        else:
            print(f"  @{off:3d}: 0x{code:04X} → [INLINE OBJ] (16B skip, params={pay[off+2:off+16].hex()})")
            off += 16
        step += 1
        if step >= 30:
            remaining = 0
            tmp = off
            while tmp + 2 <= n:
                remaining += 1
                c = struct.unpack_from('<H', pay, tmp)[0]
                tmp += 2 if (c > 0x1F or c in CTRL_2BYTE) else 16
            print(f"  ... 이하 {remaining}개 생략")
            break
    break  # 첫 번째만

---
## 9. BodyText — PARA_CHAR_SHAPE로 인라인 서식 분할

PCS(tag=68) = `(pos: uint32, char_shape_id: uint32)` × N

pos는 PARA_TEXT 안의 **문자 위치**(charCnt 단위).  
같은 char_shape_id 구간을 하나의 서식 span으로 묶으면 인라인이 됨.

In [ ]:
def extract_chars_with_pos(pt_payload):
    """PARA_TEXT에서 (charCnt 위치, 문자) 쌍 리스트를 반환."""
    result = []
    off = 0
    pos = 0
    n = len(pt_payload)
    while off + 2 <= n:
        code = struct.unpack_from('<H', pt_payload, off)[0]
        if code > 0x001F and code != 0xFFFF:
            result.append((pos, chr(code)))
            pos += 1
            off += 2
        elif code in CTRL_2BYTE:
            pos += 1
            off += 2
        else:
            pos += 8  # 인라인 오브젝트 = 8 charCnt 단위
            off += 16
    return result

def parse_pcs(pcs_payload):
    """PCS 페이로드 → [(pos, char_shape_id), ...]"""
    entries = []
    for i in range(0, len(pcs_payload) - 7, 8):
        pos, cs_id = struct.unpack_from('<II', pcs_payload, i)
        entries.append((pos, cs_id))
    return entries

def cs_id_at(entries, char_pos):
    """char_pos에 해당하는 CharShape ID 반환 (구간 검색)."""
    cs_id = entries[0][1]
    for p, cid in entries:
        if p <= char_pos:
            cs_id = cid
        else:
            break
    return cs_id

print("함수 정의 완료.")

In [ ]:
# 단락별로 PH → PT + PCS를 묶어서 인라인 분할 과정을 보여줌

para_idx = 0
i = 0
while i < len(body_recs):
    tag_id, level, ph_pay, _ = body_recs[i]
    if tag_id != 66:  # PARA_HEADER
        i += 1
        continue
    
    # 자식 레코드 수집
    children = []
    j = i + 1
    while j < len(body_recs) and body_recs[j][1] > level:
        children.append(body_recs[j])
        j += 1
    i = j
    
    pt_pay = next((p for t, l, p, o in children if t == 67), None)
    pcs_pay = next((p for t, l, p, o in children if t == 68), None)
    
    if not pt_pay:
        para_idx += 1
        continue
    
    text = extract_text(pt_pay)
    if not text.strip():
        para_idx += 1
        continue
    
    print(f"{'='*60}")
    print(f"단락 {para_idx}: \"{text}\"")
    
    if pcs_pay:
        pcs_entries = parse_pcs(pcs_pay)
        print(f"\nPCS 엔트리 ({len(pcs_entries)}개):")
        for pos, csid in pcs_entries:
            cs = char_shapes[csid] if csid < len(char_shapes) else {}
            flags = []
            if cs.get('bold'): flags.append('B')
            if cs.get('italic'): flags.append('I')
            if cs.get('underline'): flags.append('U')
            if cs.get('strikethrough'): flags.append('S')
            f = ','.join(flags) if flags else 'plain'
            print(f"  pos={pos:3d} → char_shape[{csid}] [{f}]")
        
        # 인라인 분할
        chars_pos = extract_chars_with_pos(pt_pay)
        print(f"\n인라인 분할 결과:")
        
        buf = []
        cur_csid = cs_id_at(pcs_entries, chars_pos[0][0])
        for cpos, ch in chars_pos:
            this_csid = cs_id_at(pcs_entries, cpos)
            if this_csid != cur_csid:
                cs = char_shapes[cur_csid] if cur_csid < len(char_shapes) else {}
                flags = []
                if cs.get('bold'): flags.append('B')
                if cs.get('italic'): flags.append('I')
                if cs.get('underline'): flags.append('U')
                if cs.get('strikethrough'): flags.append('S')
                f = ','.join(flags) if flags else 'plain'
                fid = cs.get('hangul_face_id', 0)
                font = face_names[fid] if fid < len(face_names) else '?'
                print(f"  ▸ \"{(''.join(buf))}\"  [{f}] font={font} size={cs.get('font_size_pt','')}pt")
                buf = [ch]
                cur_csid = this_csid
            else:
                buf.append(ch)
        
        if buf:
            cs = char_shapes[cur_csid] if cur_csid < len(char_shapes) else {}
            flags = []
            if cs.get('bold'): flags.append('B')
            if cs.get('italic'): flags.append('I')
            if cs.get('underline'): flags.append('U')
            if cs.get('strikethrough'): flags.append('S')
            f = ','.join(flags) if flags else 'plain'
            fid = cs.get('hangul_face_id', 0)
            font = face_names[fid] if fid < len(face_names) else '?'
            print(f"  ▸ \"{(''.join(buf))}\"  [{f}] font={font} size={cs.get('font_size_pt','')}pt")
    else:
        print("  PCS 없음 → 단일 서식")
    
    print()
    para_idx += 1

---
## 10. PARA_HEADER 메타데이터

```
 0-3   charCnt: uint32 (MSB=제어 플래그, 하위 30bit=문자수)
 4-5   controlMask: uint16 (절대 수정 금지!)
 8-9   para_shape_id: uint16
10     style_id: uint8
```

style_id로 Style 이름을 조회 → "개요 N" 패턴이면 Heading.

In [ ]:
import re

HEADING_RE = re.compile(r'^개요\s*(\d+)$|^Heading\s*(\d+)$', re.IGNORECASE)

para_headers = [(pay, lvl) for tid, lvl, pay, _ in body_recs if tid == 66]
print(f"PARA_HEADER: {len(para_headers)}개\n")

for idx, (pay, lvl) in enumerate(para_headers):
    print(f"--- PH[{idx}] level={lvl}, {len(pay)}B ---")
    
    # charCnt
    if len(pay) >= 4:
        raw = struct.unpack_from('<I', pay, 0)[0]
        print(f"  charCnt     = {raw & 0x3FFFFFFF} (MSB={'1' if raw & 0x80000000 else '0'})")
    
    # controlMask
    if len(pay) >= 6:
        print(f"  controlMask = 0x{struct.unpack_from('<H', pay, 4)[0]:04X}")
    
    # para_shape_id
    if len(pay) >= 10:
        ps_id = struct.unpack_from('<H', pay, 8)[0]
        align = para_shapes[ps_id].get('alignment', '?') if ps_id < len(para_shapes) else '?'
        print(f"  ps_id       = {ps_id} → align={align}")
    
    # style_id
    if len(pay) >= 11:
        sid = pay[10]
        sname = styles[sid]['name'] if sid < len(styles) else '?'
        print(f"  style_id    = {sid} → '{sname}'")
        m = HEADING_RE.match(sname.strip())
        if m:
            print(f"  ★ Heading level={int(m.group(1) or m.group(2))}")
    print()

---
## 11. PARA_LINE_SEG 확인

PLS(tag=69) = 줄 배치 정보, 36바이트/엔트리:
```
 0-3   tpos: uint32  (문자 위치 — vpos 아님!)
 4-7   vpos: uint32  (Y 좌표)
 8-11  height: uint32
20-23  line_height: uint32
```

In [ ]:
pls_recs = [(pay, lvl) for tid, lvl, pay, _ in body_recs if tid == 69]
print(f"PARA_LINE_SEG: {len(pls_recs)}개\n")

for idx, (pay, _) in enumerate(pls_recs):
    n_entries = len(pay) // 36
    print(f"--- PLS[{idx}] ({n_entries}줄) ---")
    for k in range(min(n_entries, 5)):  # 최대 5줄만
        base = k * 36
        tpos = struct.unpack_from('<I', pay, base)[0]
        vpos = struct.unpack_from('<I', pay, base + 4)[0]
        h    = struct.unpack_from('<I', pay, base + 8)[0]
        lh   = struct.unpack_from('<I', pay, base + 20)[0]
        print(f"  줄{k}: tpos={tpos:4d}  vpos={vpos:6d}  height={h:5d}  line_h={lh:5d}")
    if n_entries > 5:
        print(f"  ... ({n_entries - 5}줄 더)")
    print()

---
## 12. 표(Table) 파싱 — CTRL_HEADER + LIST_HEADER

다른 HWP 파일에서 표 구조를 확인. 표가 없으면 skip됩니다.

```
CTRL_HEADER ctrl_id='tbl '  ← 표 시작
  LIST_HEADER                ← 셀 (row, col, rowspan, colspan)
    PARA_HEADER → PARA_TEXT  ← 셀 내부 텍스트
  LIST_HEADER                ← 다음 셀
    ...
  TABLE                      ← 표 속성 (행/열 수 등)
```

In [ ]:
# 표가 있는 파일로 전환
TABLE_HWP = os.path.abspath("../workspace/05_표_셀내용.hwp")

if os.path.exists(TABLE_HWP):
    ole2 = olefile.OleFileIO(TABLE_HWP)
    fh2 = ole2.openstream('FileHeader').read()
    comp2 = bool(struct.unpack_from('<I', fh2, 36)[0] & 0x01)
    body2 = read_stream(ole2, 'BodyText/Section0', comp2)
    recs2 = parse_records(body2)
    
    print("=== 표 파일 레코드 트리 ===")
    for tid, lvl, pay, off in recs2:
        indent = '  ' * lvl
        extra = ''
        if tid == 71:
            cid = pay[:4][::-1].decode('ascii', errors='replace') if len(pay) >= 4 else '?'
            extra = f"  ctrl_id='{cid}'"
        elif tid == 72:  # LIST_HEADER
            if len(pay) >= 32:
                _, _, col, row, colspan, rowspan, sx, sy = struct.unpack_from('<IIHHHHII', pay, 0)
                extra = f"  row={row} col={col} span=({rowspan},{colspan}) size=({sx},{sy})"
        elif tid == 67:  # PARA_TEXT
            extra = f"  text=\"{extract_text(pay)}\""
        print(f"{indent}[L{lvl}] {tname(tid):20s} {len(pay):>4}B{extra}")
    
    ole2.close()
else:
    print(f"파일 없음: {TABLE_HWP}")

---
## 13. 전체 흐름 요약

```
HWP file (OLE2 compound file)
  │
  ├─ FileHeader       → flags: compressed? encrypted?
  │
  ├─ DocInfo          → zlib decompress → HWPTAG records
  │    ├─ FACE_NAME   → ["함초롬돋움", "맑은 고딕", ...]
  │    ├─ CHAR_SHAPE  → [{bold, italic, size, color}, ...]
  │    ├─ PARA_SHAPE  → [{align, margins, line_spacing}, ...]
  │    └─ STYLE       → [{name, para_shape_id, char_shape_id}, ...]
  │
  └─ BodyText/Section0 → zlib decompress → HWPTAG records
       └─ PARA_HEADER     charCnt, controlMask, para_shape_id, style_id
            ├─ PARA_TEXT       UTF-16LE + 제어코드 → 텍스트 추출
            ├─ PARA_CHAR_SHAPE (pos, cs_id) × N → 서식 구간 분할
            ├─ PARA_LINE_SEG   줄 배치 (tpos, vpos, height)
            └─ CTRL_HEADER     ctrl_id='tbl ' → 표
                 ├─ LIST_HEADER   셀 (row, col, span)
                 │    └─ PARA_HEADER → PARA_TEXT ...
                 └─ TABLE          표 속성
```

In [ ]:
ole.close()
print("완료. ole reader 닫힘.")